In [0]:
# ===== PARAMETERS =====
runs_table = "test_automation_runs2"
results_table = "test_automation_results2"
filter_states = []       # empty = all states
include_statuses = ["FAIL", "ERROR", "NO_DATA"]   # statuses considered "not passed"
# ======================

from pyspark.sql.functions import col, desc
from collections import defaultdict
import re

chain_order = [
    "paymentPending", "appealSubmitted",
    "awaitingRespondentEvidence(a)", "awaitingRespondentEvidence(b)",
    "caseUnderReview", "reasonsForAppealSubmitted", "listing",
    "prepareForHearing", "decision", "decided(a)",
    "ftpaSubmitted(a)", "decided(b)", "ftpaSubmitted(b)", "ftpaDecided",
    "remitted", "ended"
]

active_states = filter_states if filter_states else chain_order

runs_df = spark.table(runs_table)
results_df = spark.table(results_table)

trans_runs = runs_df.filter(col("run_by_automation_name") == "Transformation_Tests")
all_runs_list = trans_runs.orderBy(desc("run_start_datetime")).collect()
run_lookup = {r.run_id: r for r in all_runs_list}
all_run_ids = [r.run_id for r in all_runs_list]

if not all_run_ids:
    print(f"No runs found in {runs_table}")
else:
    all_results = results_df.filter(col("run_id").isin(all_run_ids)).toPandas()
    run_to_state = {r.run_id: r.state_under_test for r in all_runs_list}
    all_results["run_state"] = all_results["run_id"].map(run_to_state)
    all_results["status_upper"] = all_results["status"].str.upper().str.strip()

    # Reclassify on the fly (same patterns used in the main cross report)
    NO_DATA_PATTERNS = ["NO RECORDS TO TEST", "NO MATCHING TEST DATA", "DOES NOT EXIST IN THE",
                        "NO RECORDS FOUND", "NO DATA AVAILABLE", "NO DATA TO TEST",
                        "NO DATA EXISTS FOR", "UNRESOLVED_COLUMN", "FAILED TO SETUP DATA"]
    ERROR_PATTERNS = ["IS NOT DEFINED", "TEST CRASHED:"]

    def reclassify(row):
        status = row["status_upper"]
        if status == "PASS": return "PASS"
        if status in ("NO_DATA", "ERROR"): return status
        msg = str(row.get("message", "") or "").upper()
        for p in NO_DATA_PATTERNS:
            if p in msg: return "NO_DATA"
        for p in ERROR_PATTERNS:
            if p in msg: return "ERROR"
        return status

    all_results["status_upper"] = all_results.apply(reclassify, axis=1)

    # ============================================================
    # Keep only non-pass results in requested statuses and from active_states
    # ============================================================
    failures = all_results[
        all_results["status_upper"].isin([s.upper() for s in include_statuses]) &
        all_results["test_from_state"].isin(active_states)
    ].copy()

    if failures.empty:
        displayHTML("<h2>No non-pass results found</h2>"
                    f"<p>Statuses searched: {', '.join(include_statuses)}</p>"
                    f"<p>Runs scanned: {len(all_runs_list)} across {runs_table}</p>")
    else:
        # ============================================================
        # Fingerprint a message so similar failures group together.
        # Strip case refs, numbers, dates, uuids, and trim to first N chars.
        # ============================================================
        def fingerprint(msg):
            if not msg: return ""
            s = str(msg)
            # Normalise common volatile substrings
            s = re.sub(r"[A-Z]{2}/\d+/\d+", "<CASE_REF>", s)              # EA/04437/2020
            s = re.sub(r"\b[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}\b", "<UUID>", s)
            s = re.sub(r"\b\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2}(\.\d+)?\b", "<DATETIME>", s)
            s = re.sub(r"\b\d{4}-\d{2}-\d{2}\b", "<DATE>", s)
            s = re.sub(r"\b\d{3,}\b", "<N>", s)                            # long numbers
            s = re.sub(r"\s+", " ", s).strip()
            return s[:220]

        failures["fingerprint"] = failures["message"].apply(fingerprint)

        # ============================================================
        # Group by (test_name, status, fingerprint) — "similar failure"
        # ============================================================
        groups = defaultdict(lambda: {
            "count": 0,
            "states_from": set(),
            "states_run": set(),
            "runs": set(),
            "example_message": "",
            "test_fields": set(),
        })

        for _, row in failures.iterrows():
            key = (row.get("test_name", "") or "", row["status_upper"], row["fingerprint"])
            g = groups[key]
            g["count"] += 1
            g["states_from"].add(row.get("test_from_state", "") or "")
            g["states_run"].add(row.get("run_state", "") or "")
            g["runs"].add(row.get("run_id", "") or "")
            if not g["example_message"] and row.get("message"):
                g["example_message"] = str(row["message"])[:1200]
            g["test_fields"].add(str(row.get("test_field", "") or ""))

        # Sort groups: most frequent first, then by test_name
        sorted_groups = sorted(groups.items(), key=lambda x: (-x[1]["count"], x[0][0]))

        # ============================================================
        # Top-level counts
        # ============================================================
        total_failures = len(failures)
        total_fail = (failures["status_upper"] == "FAIL").sum()
        total_error = (failures["status_upper"] == "ERROR").sum()
        total_nodata = (failures["status_upper"] == "NO_DATA").sum()
        unique_groups = len(groups)

        # ============================================================
        # Build group rows with drill-down
        # ============================================================
        def sort_states(s): return sorted(s, key=lambda x: chain_order.index(x) if x in chain_order else 999)

        status_badge_class = {"FAIL": "b-fail", "ERROR": "b-err", "NO_DATA": "b-nd"}

        group_rows = ""
        for g_idx, ((tname, status, fp), g) in enumerate(sorted_groups):
            states_from_txt = ", ".join(sort_states(g["states_from"]))
            states_run_txt = ", ".join(sort_states(g["states_run"]))
            fields_txt = ", ".join(sorted(x for x in g["test_fields"] if x))
            badge = status_badge_class.get(status, "b-nd")

            snippet = g["example_message"][:160].replace("\n", " ")

            group_rows += f'''<tr class="grp-row" data-status="{status}" data-test="{tname.lower()}" data-msg="{fp.lower()}" data-fromstate="{states_from_txt}" onclick="toggleGroup({g_idx})" style="cursor:pointer">
                <td><b>{tname}</b></td>
                <td><span class="badge {badge}">{status}</span></td>
                <td style="text-align:right"><b>{g["count"]}</b></td>
                <td>{states_from_txt}</td>
                <td>{states_run_txt}</td>
                <td>{fields_txt}</td>
                <td class="msg-snippet">{snippet}</td>
            </tr>\n'''

            # Drill-down: full fingerprint + full example message + sample of individual rows
            sample_rows_html = ""
            sample_failures = failures[
                (failures["test_name"] == tname) &
                (failures["status_upper"] == status) &
                (failures["fingerprint"] == fp)
            ].head(25)
            for _, r in sample_failures.iterrows():
                sample_rows_html += f'''<tr>
                    <td>{r.get("test_from_state", "")}</td>
                    <td>{r.get("run_state", "")}</td>
                    <td>{r.get("test_field", "")}</td>
                    <td style="font-family:monospace;font-size:11px;max-width:700px">{str(r.get("message", ""))[:400]}</td>
                </tr>\n'''

            group_rows += f'''<tr class="grp-detail" id="grp-detail-{g_idx}" style="display:none">
                <td colspan="7">
                    <div style="padding:8px;background:#f8f9fa">
                        <p><b>Fingerprint:</b> <code>{fp[:500]}</code></p>
                        <p><b>Full example message:</b></p>
                        <pre style="white-space:pre-wrap;background:#fff;padding:8px;border:1px solid #ddd;max-height:240px;overflow:auto">{g["example_message"]}</pre>
                        <p><b>First {len(sample_failures)} occurrences:</b></p>
                        <table style="width:100%;font-size:11px">
                            <thead><tr>
                                <th>test_from_state</th><th>run_state</th><th>field</th><th>message</th>
                            </tr></thead>
                            <tbody>{sample_rows_html}</tbody>
                        </table>
                    </div>
                </td>
            </tr>\n'''

        # ============================================================
        # Filter dropdowns
        # ============================================================
        status_options = "".join(
            f'<option value="{s}">{s}</option>'
            for s in ["FAIL", "ERROR", "NO_DATA"]
        )
        from_state_options = "".join(
            f'<option value="{s}">{s}</option>'
            for s in sort_states({row.get("test_from_state", "") for _, row in failures.iterrows() if row.get("test_from_state")})
        )

        html = f"""
        <style>
            .csf {{font-family:Arial,sans-serif;padding:10px}}
            .csf h2 {{color:#333;border-bottom:2px solid #333;padding-bottom:5px;margin-top:25px}}
            .csf table {{border-collapse:collapse;width:100%;margin-bottom:12px;font-size:12px}}
            .csf th {{background:#2c3e50;color:white;padding:6px;text-align:left;position:sticky;top:0}}
            .csf td {{border:1px solid #ddd;padding:4px 8px;vertical-align:top}}
            .csf tr:nth-child(even) {{background:#f9f9f9}}
            .csf .grp-row:hover {{background:#e8f4fd !important}}
            .csf .badge {{display:inline-block;padding:2px 8px;border-radius:3px;color:white;font-weight:bold;font-size:11px}}
            .csf .b-fail {{background:#e74c3c}}
            .csf .b-err {{background:#922b21}}
            .csf .b-nd {{background:#f39c12}}
            .csf .box {{display:inline-block;padding:12px 20px;margin:4px;border-radius:5px;color:white;font-size:16px;text-align:center}}
            .csf .box-fail {{background:#e74c3c}}
            .csf .box-err {{background:#922b21}}
            .csf .box-nd {{background:#f39c12}}
            .csf .box-total {{background:#2c3e50}}
            .csf .msg-snippet {{font-family:monospace;font-size:11px;color:#555;max-width:500px}}
            .csf .filter-bar {{background:#ecf0f1;padding:10px;border-radius:5px;margin:10px 0;display:flex;gap:12px;align-items:center;flex-wrap:wrap}}
            .csf .filter-bar input, .csf .filter-bar select {{padding:6px 10px;border:1px solid #bdc3c7;border-radius:3px;font-size:13px}}
            .csf .filter-bar input {{min-width:240px}}
            .csf .filter-bar label {{font-weight:bold;color:#2c3e50}}
            .csf .filter-bar button {{padding:6px 14px;background:#2c3e50;color:white;border:none;border-radius:3px;cursor:pointer}}
            .csf .hidden {{display:none !important}}
            .csf #match-count {{margin-left:auto;color:#7f8c8d;font-size:12px}}
        </style>
        <div class="csf">
            <h2>Cross-State Failure Report</h2>
            <p><code>{runs_table}</code> / <code>{results_table}</code> |
               {len(all_runs_list)} runs scanned |
               Statuses: <b>{', '.join(include_statuses)}</b></p>

            <div>
                <div class="box box-fail">FAIL<br><b>{total_fail}</b></div>
                <div class="box box-err">ERROR<br><b>{total_error}</b></div>
                <div class="box box-nd">NO DATA<br><b>{total_nodata}</b></div>
                <div class="box box-total">TOTAL<br>non-pass<br><b>{total_failures}</b></div>
                <div class="box box-total">UNIQUE<br>failure groups<br><b>{unique_groups}</b></div>
            </div>

            <div class="filter-bar">
                <label>Search test/message:</label>
                <input type="text" id="filter-text" placeholder="Filter by test name or message..." oninput="applyFilters()" />
                <label>Status:</label>
                <select id="filter-status" onchange="applyFilters()">
                    <option value="">All</option>
                    {status_options}
                </select>
                <label>test_from_state:</label>
                <select id="filter-fromstate" onchange="applyFilters()">
                    <option value="">All</option>
                    {from_state_options}
                </select>
                <button onclick="clearFilters()">Clear</button>
                <span id="match-count"></span>
            </div>

            <p>Rows grouped by <b>(test_name, status, normalised message)</b>. Click any row to drill down into a sample of occurrences.</p>
            <div style="overflow-x:auto;max-height:800px;overflow-y:auto">
                <table id="grp-matrix">
                    <thead><tr>
                        <th>Test Name</th>
                        <th>Status</th>
                        <th style="text-align:right">Count</th>
                        <th>test_from_state</th>
                        <th>ran in (run_state)</th>
                        <th>Fields</th>
                        <th>Message snippet</th>
                    </tr></thead>
                    <tbody>{group_rows}</tbody>
                </table>
            </div>

            <br>
            <button onclick="downloadReport()" style="background:#2c3e50;color:white;padding:10px 25px;border:none;border-radius:5px;cursor:pointer;font-size:14px">
                Download Offline Report
            </button>
        </div>
        <script>
            function toggleGroup(idx) {{
                var row = document.getElementById('grp-detail-' + idx);
                if (row) row.style.display = row.style.display === 'none' ? 'table-row' : 'none';
            }}
            function applyFilters() {{
                var text = (document.getElementById('filter-text').value || '').toLowerCase().trim();
                var status = document.getElementById('filter-status').value;
                var fstate = document.getElementById('filter-fromstate').value;
                var rows = document.querySelectorAll('#grp-matrix tbody tr.grp-row');
                var shown = 0;
                rows.forEach(function(row) {{
                    var test = row.getAttribute('data-test') || '';
                    var msg = row.getAttribute('data-msg') || '';
                    var st = row.getAttribute('data-status') || '';
                    var fs = row.getAttribute('data-fromstate') || '';
                    var okText = !text || test.indexOf(text) !== -1 || msg.indexOf(text) !== -1;
                    var okStatus = !status || st === status;
                    var okFs = !fstate || fs.indexOf(fstate) !== -1;
                    var idx = row.getAttribute('onclick').match(/\\d+/)[0];
                    var detail = document.getElementById('grp-detail-' + idx);
                    if (okText && okStatus && okFs) {{
                        row.classList.remove('hidden');
                        shown++;
                    }} else {{
                        row.classList.add('hidden');
                        if (detail) detail.style.display = 'none';
                    }}
                }});
                document.getElementById('match-count').textContent = shown + ' of ' + rows.length + ' groups shown';
            }}
            function clearFilters() {{
                document.getElementById('filter-text').value = '';
                document.getElementById('filter-status').value = '';
                document.getElementById('filter-fromstate').value = '';
                applyFilters();
            }}
            applyFilters();
            function downloadReport() {{
                var content = document.querySelector('.csf').outerHTML;
                var styles = document.querySelectorAll('style');
                var styleText = '';
                styles.forEach(function(s) {{ styleText += s.outerHTML; }});
                var scripts = '<script>function toggleGroup(idx){{var r=document.getElementById(\"grp-detail-\"+idx);if(r)r.style.display=r.style.display===\"none\"?\"table-row\":\"none\"}}function applyFilters(){{var text=(document.getElementById(\"filter-text\").value||\"\").toLowerCase().trim();var status=document.getElementById(\"filter-status\").value;var fstate=document.getElementById(\"filter-fromstate\").value;var rows=document.querySelectorAll(\"#grp-matrix tbody tr.grp-row\");var shown=0;rows.forEach(function(row){{var t=row.getAttribute(\"data-test\")||\"\";var m=row.getAttribute(\"data-msg\")||\"\";var s=row.getAttribute(\"data-status\")||\"\";var fs=row.getAttribute(\"data-fromstate\")||\"\";var okT=!text||t.indexOf(text)!==-1||m.indexOf(text)!==-1;var okS=!status||s===status;var okF=!fstate||fs.indexOf(fstate)!==-1;var idx=row.getAttribute(\"onclick\").match(/\\\\d+/)[0];var d=document.getElementById(\"grp-detail-\"+idx);if(okT&&okS&&okF){{row.classList.remove(\"hidden\");shown++}}else{{row.classList.add(\"hidden\");if(d)d.style.display=\"none\"}}}});document.getElementById(\"match-count\").textContent=shown+\" of \"+rows.length+\" groups shown\"}}function clearFilters(){{document.getElementById(\"filter-text\").value=\"\";document.getElementById(\"filter-status\").value=\"\";document.getElementById(\"filter-fromstate\").value=\"\";applyFilters()}}<\\/script>';
                var full = '<!DOCTYPE html><html><head><meta charset=\"utf-8\"><title>Cross-State Failure Report</title>' + styleText + '</head><body>' + content + scripts + '</body></html>';
                var blob = new Blob([full], {{type: 'text/html'}});
                var a = document.createElement('a');
                a.href = URL.createObjectURL(blob);
                a.download = 'cross_state_failure_report.html';
                a.click();
            }}
        </script>
        """

        displayHTML(html)
